# ROOT TTree input with uproot

A toy is generated in one call with the default inverse-transform sampler, written to ROOT, loaded back as a `PhaseSpaceSample`, and fitted directly with `FitSession`.


In [ ]:
import numpy as np
import uproot
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel,DecayModel,FitSession,NonResonant,RealImag,
    enable_x64,generate_toy,plot_dalitz,read_phase_space_sample,
)
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [NonResonant(RealImag(1.0,0.0))],
    normalization_method="square-dalitz",normalization_resolution=120,normalization_pair=(0,2),
)
generated=generate_toy(model,15_000,seed=1313)

with uproot.recreate("b2kpipi_toy.root") as f:
    f["DecayTree"]={
        "S12":np.asarray(generated.s12),
        "S13":np.asarray(generated.s13),
        "S23":np.asarray(generated.s23),
    }

data=read_phase_space_sample(
    "b2kpipi_toy.root","DecayTree",s12="S12",s13="S13",s23="S23"
)
plot_dalitz(data,x="s13",y="s23",title="Loaded from ROOT")
plt.show()


In [ ]:
session=FitSession.from_root(
    model,"b2kpipi_toy.root","DecayTree",
    s12="S12",s13="S13",s23="S23",
)
result=session.fit()
session.report(result)
session.plot_projection(result,"s13")
plt.show()
